# Reproduce the checks and figures

This notebook reruns the parts of the project a reader can reproduce on a free Colab CPU runtime, in about 10 to 20 minutes:

1. the test suite and the SPECK32/64 known-answer test;
2. the paper figures, redrawn from the saved result files, with the published counts checked before anything is saved;
3. a short live demo: exact SAT recovery of a small reduced-round key, data generation on the demo seed, and fresh-pair key verification.

The full campaigns are not rerun here. They need multi-gigabyte difference tables and many hours of compute. See `docs/REPRODUCE.md` in the repository.

Run the cells in order. A GPU is not needed.

## Step 1: get the code

If the repository is public, run the cell as it is. If it is private, add a GitHub token with read access as a Colab secret named `GITHUB_TOKEN` (key icon on the left), and turn on notebook access for it.

In [ ]:
import os, subprocess
REPO = 'github.com/ShakChunni/confidence-gated-neural-sat-speck.git'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
if not os.path.isdir('/content/repo'):
    subprocess.run(['git', 'clone', '--depth', '1', url, '/content/repo'], check=True)
os.chdir('/content/repo')
!git log -1 --format='commit %h  %ad' --date=short

## Step 2: install

Colab already has PyTorch and NumPy. This adds the SAT solver, test tools and plotting, then installs the project.

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q --no-deps --editable .
!python -c "import sys, pycryptosat, torch, numpy, matplotlib; print(sys.version.split()[0], 'torch', torch.__version__, 'numpy', numpy.__version__, 'matplotlib', matplotlib.__version__)"

Optional: install `clang` so the tests of the native brute-force completer and the encryption calibration also run. The project builds them with clang only, as frozen in its calibration. Without it those tests skip.

In [ ]:
!apt-get -qq update && apt-get -qq install -y clang > /dev/null && clang --version | head -1

## Step 3: tests and the known-answer test

Tests that need large local files (tables, trained networks, compiled C tools) skip themselves and say why. Every other test should pass.

In [ ]:
!python -m pytest -q -p no:cacheprovider 2>&1 | tail -n 25

In [ ]:
!neural-sat-speck kat

## Step 4: redraw the paper figures

The script reads the saved result files, checks the drawn counts against the published counts, and stops before writing anything if they differ. The image bytes can differ slightly from the committed files with another matplotlib version. The input hashes and checks in `manifest.json` should not.

In [ ]:
!python scripts/generate_report_figures.py --profile report --force
!git diff --stat -- figures/report/manifest.json
!git diff -- figures/report/manifest.json | grep -E '^[-+].*(inputs_sha256|checks|count)' || echo 'inputs and checks unchanged'

In [ ]:
from IPython.display import Image, display
import glob
for path in sorted(glob.glob('figures/report/*.png')):
    print(path)
    display(Image(path, width=700))

## Step 5: short live demo (demo seed 999999)

Exact SAT recovery of the full key at 2, 4 and 7 rounds from 4 known pairs, with a 10 s limit per instance, and each key checked on fresh pairs. Seven rounds can time out with only 4 pairs. That is an expected result, not an error.

In [ ]:
import os
os.environ['PYTHONPATH'] = '.:src'
!python scripts/phase2_sat_pilot.py --rounds 2 4 7 --pair-count 4 --timeout 10 | python -c "import json,sys; [print(r['rounds'], 'rounds:', r['status'], 'verified' if r['verified'] else 'not verified', f\"{r['runtime_seconds']:.2f} s\") for r in json.load(sys.stdin)]"

In [ ]:
!neural-sat-speck generate-data --instances 2 --pairs 64 --rounds 6 --seed 999999 --output /tmp/demo_instances.json
!python -c "import json; d=json.load(open('/tmp/demo_instances.json')); print(type(d).__name__, list(d)[:8] if isinstance(d, dict) else len(d))"

## What this does not reproduce

- The 300 s attack campaigns (D-031, D-041, D-060, D-063 to D-066). They need the 4 to 32 GiB difference tables, Gohr's and Bao's published networks, and about 7 to 10 hours per campaign on one laptop.
- Exact timings. They depend on the machine.

The saved ledgers behind every figure are in the repository, with the commands and hashes in `docs/REPRODUCE.md`.